In [1]:
import json

from protest_impact.util import project_root

with open(
    project_root / "data" / "news" / "scrapable_mediacloud_newspapers_v1.json"
) as f:
    newspapers = json.load(f)

In [2]:
newspapers[:5]

[{'name': 'hna', 'media_id': 39267},
 {'name': 'tz.de', 'media_id': 143671},
 {'name': 'merkur-online', 'media_id': 39005},
 {'name': 'Christ & Welt: ZEIT für Glaube, Geist und Gesellschaft',
  'media_id': 385524},
 {'name': 'wn.de', 'media_id': 145368}]

In [3]:
from datetime import datetime
from os import environ
from shutil import rmtree

import pandas as pd
from dotenv import load_dotenv

from protest_impact.util.cache import get

load_dotenv()


def get_counts(newspaper, query):
    media_id = newspaper["media_id"]
    response = get(
        "https://api.mediacloud.org/api/v2/stories_public/count/",
        params={
            "q": query,
            "fq": f"media_id:{media_id}",
            "split": True,
            "split_period": "day",
            "key": environ["MEDIACLOUD_API_KEY"],
        },
        headers={"Accept": "application/json"},
    )
    result = response.json()
    if "counts" not in result:
        return None
    #     hash_ = get._get_argument_hash(
    #         "https://api.mediacloud.org/api/v2/stories_public/count/",
    #         params={
    #             "q": query,
    #             "fq": f"media_id:{media_id}",
    #             "split": True,
    #             "split_period": "day",
    #             "key": environ["MEDIACLOUD_API_KEY"],
    #         },
    #         headers={"Accept": "application/json"},
    #         )
    #     # delete the cache folder
    #     rmtree(project_root / ".cache" / "protest_impact" / "util" / "cache" / "get" / hash_)
    #     return None
    df = pd.DataFrame(result["counts"])
    if len(df) == 0:
        return None
    df["date"] = pd.to_datetime(df["date"])
    df = df[(df["date"].dt.year >= 2010) & (df["date"] < datetime.now())]
    df["media_id"] = media_id
    df["name"] = newspaper["name"]
    return df

In [4]:
import yaml

with open(project_root / "protest_impact" / "data" / "protests" / "config.yaml") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [5]:
movement_kws = config["movement_keywords"]
test_kw = movement_kws["climate"]["movement"][0]
test_kw

'fridays for future'

In [6]:
def querify(s):
    if len(s.split()) == 1:
        return s
    else:
        return f'"{s}"'

In [7]:
querify("fridays for future")

'"fridays for future"'

In [8]:
get_counts(newspapers[0], querify(test_kw)).head()

,count,date,media_id,name
0,1,2018-12-14,39267,hna
1,1,2019-01-31,39267,hna
2,1,2019-02-02,39267,hna
3,1,2019-03-03,39267,hna
4,1,2019-03-14,39267,hna


In [10]:
from tqdm.notebook import tqdm

dfs = []
for movement in list(movement_kws):
    print(movement)
    if "topic" not in movement_kws[movement]:
        continue
    for kw in tqdm(list(movement_kws[movement]["topic"])):
        for newspaper in newspapers:
            counts = get_counts(newspaper, querify(kw))
            if counts is not None:
                dfs.append(counts)
df = pd.concat(dfs)
df.to_csv("topic_counts.csv", index=False)
df.head()

climate


  0%|          | 0/10 [00:00<?, ?it/s]

racism


  0%|          | 0/9 [00:00<?, ?it/s]

feminism


...

  0%|          | 0/4 [00:00<?, ?it/s]

international


  0%|          | 0/7 [00:00<?, ?it/s]

political-parties


,count,date,media_id,name
0,1,2013-03-13,39267,hna
1,1,2013-03-24,39267,hna
2,1,2013-03-29,39267,hna
3,1,2013-04-04,39267,hna
4,1,2013-04-15,39267,hna
